# ComfyUI مجاني على Kaggle — توليد فيديو بدون أي مصاريف

**قبل التشغيل (مرة واحدة فقط):**
1. من قائمة **Settings** أعلى اليمين: `Accelerator` = **GPU T4 x2 أو P100** و `Internet` = **On**
2. اضغط **Run All** وانتظر حتى تظهر الخلية الأخيرة رابط الوصول
3. افتح الرابط في متصفحك → ComfyUI جاهز

**الخطة:** Wan 2.1 1.3B (صغير وسريع على T4) + رفع الدقة Real-ESRGAN + نفق Cloudflare مجاني.
جلسة واحدة تصل إلى ~9 ساعات، ولديك 30 ساعة GPU أسبوعياً.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
# P100 (sm_60): cu128 dropped sm_60; torch 2.10.0+cu126 still ships sm_60 kernels (verified on Kaggle P100)
# Pin ALL of torch/torchvision/torchaudio to matching versions; unpinned ones cause ABI mismatch
!pip uninstall -q -y torch torchvision torchaudio
!pip install -q --no-cache-dir torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126
!pip install -q huggingface_hub einops transformers accelerate sentencepiece tokenizers peft
import torch
print('TORCH:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os
print('STEP 3: downloading Wan 2.1 models')

base = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
files = [
    ('split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors', '/kaggle/working/ComfyUI/models/diffusion_models'),
    ('split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', '/kaggle/working/ComfyUI/models/text_encoders'),
    ('split_files/vae/wan_2.1_vae.safetensors', '/kaggle/working/ComfyUI/models/vae'),
]
for src, dst in files:
    out = hf_hub_download(repo_id=base, filename=src, local_dir='/kaggle/working/_hf')
    shutil.move(out, os.path.join(dst, os.path.basename(src)))
    print('OK:', os.path.basename(src))

In [ ]:
import shutil, os, urllib.request, ssl

# Real-ESRGAN رفع الدقة (تجرب عدة مصادر)
os.makedirs('/kaggle/working/ComfyUI/models/upscale_models', exist_ok=True)
dst = '/kaggle/working/ComfyUI/models/upscale_models/RealESRGAN_x4plus.pth'
urls = [
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth',
]
ok = False
for u in urls:
    try:
        urllib.request.urlretrieve(u, dst)
        ok = True
        print('OK: RealESRGAN from', u.split('/')[2])
        break
    except Exception as e:
        print('failed:', u, '-', e)
        continue
if not ok:
    print('WARNING: upscaler download failed (optional, continue without it)')

try:
    os.makedirs('/kaggle/working/ComfyUI/user/default/workflows', exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/text_to_video_wan.json',
        '/kaggle/working/ComfyUI/user/default/workflows/text_to_video_wan.json')
    print('OK: workflow text_to_video_wan')
except Exception as e:
    print('WARNING: workflow download failed:', e)

In [ ]:
import os, time, subprocess, sys, urllib.request
os.chdir('/kaggle/working/ComfyUI')
log = open('/kaggle/working/comfyui.log', 'w')
proc = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--disable-auto-launch'],
                        stdout=log, stderr=subprocess.STDOUT)
print('ComfyUI PID:', proc.pid)
up = False
for i in range(20):
    time.sleep(15)
    try:
        urllib.request.urlopen('http://localhost:8188/system_stats', timeout=10)
        up = True
        print('ComfyUI is UP after', (i + 1) * 15, 'seconds')
        break
    except Exception:
        if proc.poll() is not None:
            print('ComfyUI process EXITED with code', proc.returncode)
            break
if not up:
    print('ComfyUI did not respond in time')
print('----- comfyui.log tail -----')
print(open('/kaggle/working/comfyui.log').read()[-4000:])

In [ ]:
import os, time, urllib.request, subprocess
os.chdir('/kaggle/working')
urllib.request.urlretrieve(
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '/kaggle/working/cloudflared')
os.chmod('/kaggle/working/cloudflared', 0o755)
log = open('/kaggle/working/tunnel.log', 'w')
proc = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://localhost:8188', '--no-autoupdate'],
                        stdout=log, stderr=subprocess.STDOUT)
print('cloudflared PID:', proc.pid)
time.sleep(25)
print(open('/kaggle/working/tunnel.log').read()[-2000:])

In [ ]:
import re, urllib.request
log = open('/kaggle/working/tunnel.log').read()
urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
cui = open('/kaggle/working/comfyui.log').read()[-3000:]
try:
    urllib.request.urlopen('http://localhost:8188/system_stats', timeout=10)
    health = 'UP'
except Exception:
    health = 'DOWN'
msg = f"URL={urls[0] if urls else 'NONE'}\nCOMFYUI={health}\nLOGTAIL={cui}"
print(msg[:3000])
try:
    req = urllib.request.Request('https://ntfy.sh/kaggle-comfyui-7f3a9c2e', data=msg.encode(), method='POST')
    urllib.request.urlopen(req, timeout=30)
    print('NTFY: sent')
except Exception as e:
    print('NTFY failed:', e)

## الاستخدام
- في المتصفح: قائمة **Workflows** (أعلى يسار) → `text_to_video_wan`
- اختر النموذج `wan2.1_t2v_1.3B_fp16.safetensors` إن طُلب
- اكتب الوصف في عقدة CLIP Text Encode (بالإنجليزية، مفصلاً)
- **رفع الدقة:** أضف عقدة `UpscaleImage (using Model)` واختر `RealESRGAN_x4plus` بين VAE Decode و Save
- **تنزيل النتيجة:** اضغط بزر الفأرة الأيمن على عقدة Save → Save ، ثم افتح تبويب **Output** في Kaggle لتنزيله أو استخدم `kaggle kernels output download`

### نصائح الجلسة
- لا تغلق التبويب: الجلسة تنتهي بعد ~9 ساعات أو عند الخمول
- احفظ أي نتيجة فوراً في جهازك، فالقرص يُمسح بعد الجلسة
- إذا أعطى النموذج ذاكرة منخفضة: أضف `--lowvram` إلى أمر التشغيل في خلية التشغيل

In [ ]:
# اختياري: LTX-Video 2B (جودة أعلى، ~9.4GB إضافية على القرص)
# من Hugging Face: Lightricks/LTX-Video → ltx-video-2b-v0.9.safetensors إلى models/checkpoints
# from huggingface_hub import hf_hub_download
# hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltx-video-2b-v0.9.safetensors',
#                local_dir='/kaggle/working/ComfyUI/models/checkpoints')

## الخلية الأخيرة: إبقاء الخادم حيًا
لا تغير هذه الخلية ولا توقفها أثناء العمل — بقاء الجلسة حية يبقي ComfyUI متصلاً ورابط النفق عملاً حتى نهاية الجلسة (~9 ساعات).

In [ ]:
import time
while True:
    time.sleep(300)